# Retraining the AMIGO model

This notebook continues the training of AMIGO, the forward model of JWST/NIRISS aperture masking
interferometry (AMI), on calibration data. The model has three parts, fitted together:

- **the optics**: where the mask's seven holes sit and how each is warped, and the wavefront (optical path
  difference, OPD) across them. Together they give the point-spread function (PSF).
- **the detector**: each pixel's sensitivity (the flat field, `FF`), non-linearity and dark current, and the blur
  of the PSF on the pixels.
- **a small neural network (NN)** that predicts how charge migrates between neighbouring pixels as they fill up
  during an exposure.

The data are calibrator stars, a binary star, detector flats and darks, and "validators": calibrator stars from
other programs. The validators only fit their own parameters (position, brightness, ...), never the shared model,
so their loss measures how well the model carries over to data it is not trained on.

The run starts from an already trained state (the 0.2 base model) and trains it further.

In [ ]:
import os
import sys
import shutil
import importlib.resources
from datetime import datetime
from glob import glob

import jax

jax.config.update("jax_enable_x64", True)  # everything in double precision

import jax.numpy as np
import jax.random as jr
import jax.tree as jtu
import numpy as onp
import optax

import amigo
import dLux
import zodiax

import matplotlib.pyplot as plt
import scienceplots  # noqa: F401 (registers the "science" style)

plt.style.use(["science", "bright", "no-latex"])
plt.rcParams["image.cmap"] = "inferno"
plt.rcParams["font.family"] = "serif"
plt.rcParams["image.origin"] = "lower"
plt.rcParams["font.size"] = 8
plt.rcParams["ytick.direction"] = "out"

device = jax.local_devices()[0]
print(device.device_kind, "| 64bit" if jax.config.read("jax_enable_x64") else "| 32bit")
print("jax", jax.__version__, "| zodiax", zodiax.__version__, "| dLux", dLux.__version__, "| amigo", amigo.__version__)

if device.device_kind != "cpu":
    # Compiled programs are cached on /fred, shared by every job and node: compiling takes ~9 minutes at the
    # start of a run, and with the same exposures and network it is the same work every time
    jax.config.update("jax_compilation_cache_dir", "/fred/oz440/max/data/jax_cache")
    jax.config.update("jax_persistent_cache_min_entry_size_bytes", -1)
    jax.config.update("jax_persistent_cache_min_compile_time_secs", 0)
    jax.config.update("jax_persistent_cache_enable_xla_caches", "xla_gpu_per_fusion_autotune_cache_dir")
    stats = device.memory_stats()
    print(f"GPU memory {stats['bytes_limit'] / 1024**3:.2f} GB, in use {stats['bytes_in_use'] / 1024**3:.2f} GB")

## Settings

In [ ]:
# Where the instrument differs between primary dither positions: focus and rotation are fitted once per program
# (and filter) plus a constant for each position (True), or once per program, filter and position (False)
dither_flag = True

# Where everything lives
data_dir = "/fred/oz440/max/data/JWST/"  # the calslope files, one folder per program
calslope_folder = "calslope_amigo-0.2"  # in each program's folder: made with x64 and the INL ADC correction
training_dir = "/fred/oz440/max/data/amigo_training/v0.2"  # the 0.2 base model's training files
output_path = "/fred/oz440/max/outputs/retrain"  # one folder per run (interactive runs: scratch/)
fisher_dir = "/fred/oz440/max/outputs/degeneracy"  # the Fisher caches

# The trained state to start from: the 0.2 base model (32 x 3 network, trained with CAL 1509)
best_state_path = os.path.join(training_dir, "retrain_state_dither.npy" if dither_flag else "retrain_state.npy")
# The wavefront modes the state's coefficients refer to (Fourier, Zernike and piston/tip/tilt)
base_opds_path = os.path.join(training_dir, "base_opds.npz")
epochs = 1000
nn_config = 12  # network shape, index into retrain_fns.nn_setup_options: 12 = 32 wide, 3 hidden layers
# Fisher matrices are cached by exposure and parameter name only, so a folder belongs to one likelihood and one set
# of wavefront modes; any that are missing are computed (slow). A run rewrites every Fisher it loads, so give each
# run its own copy of the folder (`cp -r`, not hard links, which share the files).
fisher_cache = os.path.join(fisher_dir, "linear_model_v2_v02_d32_1509", "cache")
recalculate = False  # True recomputes every Fisher and overwrites the cached ones
# Which data to train on
val_flag = True  # the validators
binary_flag = True  # the GO 8330 binary
flat_flag = True  # the detector flats
dark_flag = True  # the detector darks
init_plots = False  # plot every exposure's fit at the start

## Paths and the run folder
Run as a script, everything this file prints and saves goes to a new folder in `output_path`, named by the start
time and, for a Slurm job, the job's name (`sbatch -J <name>`; the validation suite finds runs by it). Run
interactively, no run folder is made (but see the Saving section).

In [ ]:
from retrain_fns import Tee


def check_script():
    if "__file__" in globals():
        print("Running as a script")
        return True
    print("Running in a notebook or interactive shell")
    return False


save_flag = check_script()
if save_flag:
    run_name = datetime.now().strftime("%y-%m-%d_%H-%M-%S_%f")
    if "SLURM_JOB_NAME" in os.environ:  # Slurm's record of the job, not a setting
        run_name += "_" + os.environ["SLURM_JOB_NAME"]
    save_path = os.path.join(output_path, run_name)
    os.makedirs(save_path, exist_ok=True)
    shutil.copy(__file__, os.path.join(save_path, "script.py"))
    # from here on, everything printed (errors too) also goes to output.txt
    sys.stdout = Tee(os.path.join(save_path, "output.txt"))
    sys.stderr = sys.stdout
else:
    save_path = None
    # interactive runs keep their latest states here (see Saving)
    scratch_path = os.path.join(output_path, "scratch")
    os.makedirs(scratch_path, exist_ok=True)
print(f"Save path: {save_path}")

# Data
Every exposure is a `calslope` file: for each pixel, the slope of its ramp between consecutive reads (the signal
collected per group) and the covariance of those slopes.

Each file's bad-pixel map is replaced by amigo's static one, plus pixel [25, 74], which flares in the darks and in
GO 8330. `sort_files` keeps up to `n` files per filter. Calibrators and validators were observed at one of the AMI
dither positions; only the two primary ones (POS1, POS2) are kept, alternating between them. Flats and darks are lab
exposures with no dither position, so they are not split by it.

In [ ]:
from amigo.files import get_files
from amigo.misc import primary_offset
from amigo.model_fits import REFERENCE_OFFSET
from retrain_fns import summarise_files

with importlib.resources.as_file(importlib.resources.files(amigo).joinpath("data/badpix.npy")) as file_path:
    badpix = np.array(np.load(file_path), dtype=int)
badpix_bool = badpix.astype(bool)  # for masking arrays; the integer map is what the FITS files hold


def add_badpix(file):
    file["BADPIX"].data = badpix.copy()
    file["BADPIX"].data[25, 74] = 1
    return file


def sort_files(files, n, use_pos=True):
    files.sort(
        key=lambda x: (x[0].header["PROGRAM"], x[0].header["FILTER"], x[0].header["PATT_NUM"]),
        reverse=True,
    )
    kept = []
    for filt in ["F480M", "F430M", "F380M"]:
        matched = [f for f in files if f[0].header["FILTER"] == filt]
        if use_pos:
            pos1 = [f for f in matched if primary_offset(f[0].header) == REFERENCE_OFFSET]
            pos2 = [f for f in matched if primary_offset(f[0].header) != REFERENCE_OFFSET]
            matched = [f for pair in zip(pos1, pos2) for f in pair] + pos1[len(pos2) :] + pos2[len(pos1) :]
        kept += matched[:n]
    return kept

## Validators
Calibrator stars from three other programs (01093, 01242, 01843), up to five exposures per filter.

In [ ]:
val_files = []
if val_flag:
    val_files = get_files(os.path.join(data_dir, "VAL_SUBSET", calslope_folder, ""), "nis_calslope", IS_PSF=True)
    val_files = [add_badpix(file) for file in sort_files(val_files, 5)]
    summarise_files(val_files)

## Calibrators
The training data: CAL 4481 (the single star HD 41094, five dithers in each filter), GO 8330 (the binary EZ Aqr,
ten exposures per filter at two dither positions; its exposures with three or fewer groups are left out) and CAL 1509
observations 002 and 003 (HD-93237 and HR-7258, F480M only, all four dither positions). CAL 1509 is a pixel-linearity
program: its peaks reach 44-48k DN, where the other F480M data stop near 31k, so it shows the network how charge
moves in deep F480M exposures. Its observation 001 (HD-105521) is kept out as a test.

In [ ]:
print("CAL 4481 - Anand Sivaramakrishnan")
calpsf_files = get_files(os.path.join(data_dir, "4481", calslope_folder, ""), "nis_calslope", IS_PSF=True)
calpsf_files = [add_badpix(file) for file in sort_files(calpsf_files, 10)]
summarise_files(calpsf_files)

calbin_files = []
if binary_flag:
    print("\nGO 8330 - Benjamin Pope")
    calbin_files = get_files(os.path.join(data_dir, "8330", calslope_folder, ""), "nis_calslope")
    calbin_files = [file for file in calbin_files if file[0].header["NGROUPS"] > 3]
    calbin_files = [add_badpix(file) for file in sort_files(calbin_files, 10)]
    summarise_files(calbin_files)

print("\nCAL 1509 - Kevin Volk")
cal1509_files = get_files(os.path.join(data_dir, "1509", calslope_folder, ""), "nis_calslope", IS_PSF=True)
cal1509_files = [add_badpix(file) for file in cal1509_files if file[0].header["OBSERVTN"] in ("002", "003")]
for file in cal1509_files:
    # HR-7258's brightest pixels have non-finite slope covariances: mask every pixel without finite data and positive
    # variances (one would turn the gradients into NaN)
    cov = onp.asarray(file["SLOPE_COV"].data, float)
    finite = onp.isfinite(file["SLOPE"].data).all(0) & onp.isfinite(cov).all((0, 1))
    file["BADPIX"].data[~(finite & (onp.diagonal(cov, axis1=0, axis2=1) > 0).all(-1))] = 1
summarise_files(cal1509_files)
calpsf_files += cal1509_files  # every dither position is kept

## Flats and darks
Lamp flats light the whole detector smoothly, so they train the pixel sensitivities, the non-linearity and, through
extended structure the stars never show, the network. The darks (no light) train the dark current. The 800-group
darks are left out.

In [ ]:
flat_files, dark_files = [], []
if flat_flag:
    print("\nFLATS")
    flat_files = get_files(os.path.join(data_dir, "FLATS", calslope_folder, ""), "nis_calslope", EXP_TYPE="NIS_LAMP")
    flat_files = [add_badpix(file) for file in sort_files(flat_files, 10, use_pos=False)]
    summarise_files(flat_files)
if dark_flag:
    print("\nDARKS")
    dark_files = get_files(os.path.join(data_dir, "DARKS", calslope_folder, ""), "nis_calslope", EXP_TYPE="NIS_DARK")
    dark_files = [file for file in dark_files if file[0].header["NGROUPS"] != 800]
    dark_files = [add_badpix(file) for file in sort_files(dark_files, 10, use_pos=False)]
    summarise_files(dark_files)

# Exposures and shared parameters
Each file becomes an *exposure*: an object that models its own data from the model's parameters. Some parameters
belong to a single exposure (the star's position and brightness), others are shared by a group of exposures. An
exposure finds its value of a parameter through a *key*: exposures that return the same key share that value, and
training sums their gradients. Choosing the keys decides what the data are allowed to tell apart.

| parameter | what it is | one value per |
|---|---|---|
| `ptt_coeffs` | each hole's piston, tip and tilt (17 terms) | calibrators: program, filter and primary dither position; validators: exposure |
| `fourier_coeffs` | high-order wavefront (Fourier modes in every hole) | calibrator program |
| `zernike_coeffs` | low-order wavefront | filter |
| `defocus` | focus | program (observation for 01093) and filter, plus a constant per dither position |
| `rotations` | small rotation of the pupil | program (observation for 01093), plus a constant per dither position and the wheel position |
| `dither_rotation`, `dither_defocus` | the rotation and focus each primary dither position adds to the (0, 0) position | position |
| `sigmas` | blur of the PSF on the detector (mostly charge diffusion) | filter |
| `positions`, `fluxes` | the star's position and brightness | exposure |
| `spectra` | the star's spectral slope | star and filter |

Why these choices:

- **Fourier set**: the high-order wavefront changes over time, so each calibrator program fits its own and every
  other program uses that of the calibrator program observed closest in time (median `DATE-OBS`), among those
  observed in all three filters. CAL 1509's set is fitted on F480M alone and does not carry over to the other
  filters (validators in F380M fit worse with it), so only 1509 itself uses it.
- **Focus**: it differs between programs, between 01093's visits (weeks apart: ~40 um) and between dither positions
  (by 18-38 um). The position's part is the same for every filter and program, so it is one constant per position
  (`dither_defocus`, fitted from the calibrators; the (0, 0) position has none), and each program (01093: each
  observation) and filter fits one focus at (0, 0). A target at a position no calibrator visited gets the constant.
  With `dither_flag = False`, each program, filter and position fits its own instead. Positions are labelled by the
  exposure's commanded offset (`amigo.misc.primary_offset`, to the nearest 0.5 arcsec), not by `get_pos`, which
  gives two different positions the same name.
- **Rotation**: the mask sits in the NIRISS pupil wheel, which does not return to exactly the same angle (header
  `PWCPOS`, in 0.00244 deg steps, spread over 0.3 deg between visits). Each program fits a small rotation on top of
  the detector's fixed one; 01093's validators come from visits at different wheel positions, so it fits one per
  observation. Over 214 calibrator exposures the rotation follows the wheel one for one (slope -0.97 +- 0.04), so
  every exposure also shifts it by minus how far its wheel angle is from the mean of the exposures sharing its key
  (fixed, not fitted), and each dither position adds a constant (`dither_rotation`, up to 0.014 deg).
- **Blur**: charge diffusion depends on how deep the photons are absorbed, so on wavelength. With one blur for all
  filters the network bends to make up the difference, so each filter has its own. Only the calibrators fit them;
  validators use their filter's.
- **Validators** fit their own piston/tip/tilt, focus, rotation, position, brightness and spectrum; everything
  else they read from the calibrators.

In [ ]:
from amigo.model_fits import PointFit, BinaryFit, FlatFit, DarkFit

cal_files = calpsf_files + calbin_files

# Median observing date of every program, and the calibrator program observed closest in time to each
program_dates = {}
for file in cal_files + val_files:
    program_dates.setdefault(file[0].header["PROGRAM"], []).append(
        onp.datetime64(file[0].header["DATE-OBS"]).astype(float)
    )
program_dates = {prog: onp.median(days) for prog, days in program_dates.items()}
fourier_programs = sorted({file[0].header["PROGRAM"] for file in cal_files})
# the calibrator programs observed in every filter: only their Fourier sets serve other programs
all_filters = {"F380M", "F430M", "F480M"}
shared_fourier_programs = [
    prog for prog in fourier_programs
    if {file[0].header["FILTER"] for file in cal_files if file[0].header["PROGRAM"] == prog} == all_filters
]
nearest_fourier_program = {
    prog: min(shared_fourier_programs, key=lambda cal: abs(program_dates[cal] - date))
    for prog, date in program_dates.items()
}
print("Fourier set per calibrator program", fourier_programs, "; others use",
      {p: c for p, c in nearest_fourier_program.items() if p not in fourier_programs})


def position(exp):
    """Where the exposure sits on the detector: its primary dither position."""
    return exp.primary_offset if dither_flag else exp.POS


def shared_key(exp, param):
    """The key of a shared parameter, as in the table above (None: amigo's own key)."""
    match param:
        case "fourier_coeffs":
            return exp.program if exp.program in fourier_programs else nearest_fourier_program[exp.program]
        case "zernike_coeffs" | "sigmas":
            return exp.filter
        case "defocus":
            if dither_flag:  # 01093's visits, weeks apart, differ by ~40 um: its focus follows the rotation's key
                return "_".join((shared_key(exp, "rotations"), exp.filter))
            return "_".join((exp.program, exp.filter, exp.POS))
        case "rotations":
            return "_".join((exp.program, exp.observation)) if exp.program == "01093" else exp.program
    return None


class CalibratorFit(PointFit):
    def get_key(self, param):
        if param == "ptt_coeffs":
            return "_".join((self.program, self.filter, position(self)))
        return shared_key(self, param) or super().get_key(param)


class BinaryCalibratorFit(BinaryFit):
    def get_key(self, param):
        if param == "ptt_coeffs":
            return "_".join((self.program, self.filter, position(self)))
        return shared_key(self, param) or super().get_key(param)


class ValidatorFit(PointFit):
    def get_key(self, param):
        if param == "ptt_coeffs":
            return "_".join((self.filter, self.key))
        return shared_key(self, param) or super().get_key(param)

The likelihood treats every slope of a pixel as independent, using only the diagonal of the slope covariance. This
is the likelihood the 0.2 model was trained with and that the cached Fishers belong to (neighbouring slopes share a
read, so their noise is really anti-correlated, but the full covariance gives losses and Fishers that cannot be
compared with these). The flats go through the full model, network included: they are what teaches the network
about extended structure.

In [ ]:
calpsf_exposures = [CalibratorFit(file, covariance="diagonal") for file in calpsf_files]
calbin_exposures = [BinaryCalibratorFit(file, covariance="diagonal") for file in calbin_files]
val_exposures = [ValidatorFit(file, covariance="diagonal") for file in val_files]
flat_exposures = [FlatFit(file, covariance="diagonal") for file in flat_files]
dark_exposures = [DarkFit(file, covariance="diagonal") for file in dark_files]

cal_exposures = calpsf_exposures + calbin_exposures
if dither_flag:
    # The pupil wheel's reported angle (PWCPOS) moves the rotation one for one (slope -1 over 214 calibrator
    # exposures): each exposure reads its key's fitted rotation minus how far its wheel angle is from the mean of
    # the exposures sharing that key
    rotation_groups = {}
    for exp in cal_exposures + val_exposures:
        rotation_groups.setdefault(exp.get_key("rotations"), []).append(exp)
    for group in rotation_groups.values():
        mean_pwcpos = onp.mean([exp.pwcpos for exp in group])
        for exp in group:
            object.__setattr__(exp, "rotation_offset", -(exp.pwcpos - mean_pwcpos))
exposures = cal_exposures + val_exposures + flat_exposures + dark_exposures
print(f"Total number of exposures: {len(exposures)}")
_ = [f.close() for f in cal_files + val_files + flat_files + dark_files]

Training goes through the exposures in batches of three. The batch names matter to the trainer: the network is
updated after every calibrator (`cal`) and flat (`flat`) batch, and from validator (`val`) batches only the
validators' own parameters keep their gradients.

In [ ]:
from amigo.legacy.fitting import batch_exposures

cal_batch = batch_exposures(cal_exposures, batch_size=3, key="cal")
flat_batch = batch_exposures(flat_exposures + dark_exposures, batch_size=3, key="flat")
val_batch = batch_exposures(val_exposures, batch_size=3, key="val")
n_batch = len(cal_batch) + len(flat_batch)  # the batches the network learns from

# The model
The optics follow amigo's pipeline design. The holes sit at fitted positions (`hole_positions`, relative to a
common shift that is held fixed) and each has its own polynomial warp (`hole_distortion`). The wavefront across
them is the sum of three parts, whose modes come from `base_opds_path`: a Fourier set (`fourier_coeffs`), a
Zernike set (`zernike_coeffs`) and every hole's piston, tip and tilt (`ptt_coeffs`).

The detector model rotates, resamples and blurs the PSF onto the pixels. The ramp model then builds up the charge
in every pixel through the exposure, with the network moving charge between neighbours, and the read model adds
the non-linearity and dark current.

The trained state is loaded first: the aperture takes the warp's order and the holes' common shift from it.
States saved before amigo's renaming of the wavefront parts call them `fourier_opd`, `filter_opd` and
`aberrations`; they are renamed on loading.

In [ ]:
from amigo.core_models import AmigoModel
from amigo.optical_models import AMIOptics, AffineDistortion, DynamicApertureMask, PolynomialDistortion
from amigo.detector_models import LinearDetector
from amigo.ramp_models import NonLinearRamp
from amigo.read_models import ReadModel
from amigo.opd import load_opds
from retrain_fns import nn_setup_options

OLD_WF_NAMES = {"fourier_opd": "fourier_coeffs", "filter_opd": "zernike_coeffs", "aberrations": "ptt_coeffs"}


def migrate_wf_names(state):
    if "fourier_opd" in state:
        for old, new in OLD_WF_NAMES.items():
            if old in state:
                state[new] = state.pop(old)
    return state


# every value as a jax array (with its own dtype), however the state was saved
best_state = migrate_wf_names(jtu.map(np.asarray, onp.load(best_state_path, allow_pickle=True).item()))

opds = load_opds(base_opds_path)
# the warp's order from its number of terms per axis: 2 at order 1 (affine), 5 at order 2, 9 at order 3
hole_order = {2: 1, 5: 2, 9: 3}[np.shape(best_state["hole_distortion"])[-1]]
pupil_mask = DynamicApertureMask(
    distortion=AffineDistortion() if hole_order == 1 else PolynomialDistortion(order=hole_order),
    opds=opds,
    # the names the fitted parameters go by, and where they sit in the aperture
    alias={
        **{f"{name}_coeffs": f"opds.{name}.x.x" for name in opds},
        "hole_positions": "holes.x",
        "hole_distortion": "distortion.coefficients",
    },
)
pupil_mask = pupil_mask.add("holes.b", np.asarray(best_state["hole_shift"]))
# all wavelengths are propagated at once
optics = AMIOptics(pupil_mask=pupil_mask, sparse=True, psf_interpolation="intensity", wavelength_batch_size=1024)
print(f"Hole warp of order {hole_order}")

model = AmigoModel(
    exposures=exposures,
    optics=optics,
    detector=LinearDetector(),
    ramp_model=NonLinearRamp(**nn_setup_options[nn_config]),
    read=ReadModel(per_pixel_dark=True),
)
print(nn_setup_options[nn_config], model.nn_weights.shape)

# Starting from a trained state
The model starts from amigo's first guesses; now every parameter is copied from the state. For parameters with one
value per key, the state's values replace the model's key by key. Only the wavefront and focus keys some
exposure reads are kept, so no unused set gets an optimiser or a Fisher.

Then a few fixed choices:

- the flat field is defined relative to its median (which is set to 1);
- bad pixels get sensitivity 1, no non-linearity and the median dark current (during training they are reset
  after every update, or get no gradient, see below);
- the calibrators' spectral slopes are held at values from `expecto.ipynb` (for EZ Aqr: A, which includes C, and B);
  the validators' are fitted;
- for the binary, `contrasts` is the brighter star A's share of the flux. Swapping A and B (position angle +180
  deg, contrast 1 - c) hardly changes the data, so a state labelled the other way round would put A's spectrum on B.
  Started with A brighter, a fit cannot cross 0.5.

In [ ]:
state_params = [
    "positions", "fluxes", "spectra", "flat_coeffs", "pas", "separations", "contrasts",  # per exposure or star
    "ptt_coeffs", "fourier_coeffs", "zernike_coeffs", "defocus", "rotations", "sigmas",  # optics, per key
    "hole_positions", "hole_distortion", "sigma",  # optics and detector blur, global
    "nn_weights", "FF", "non_linearity", "dark_current",  # the network and the pixel maps
]
if not flat_flag:
    state_params.remove("flat_coeffs")
if not binary_flag:
    state_params = [p for p in state_params if p not in ("pas", "separations", "contrasts")]

for key, value in best_state.items():
    if key not in state_params:
        continue
    if key in model.params.keys():
        model = model.set(key, model.get(key) | value)
    else:
        model = model.set(key, value)


def read_keys(param):
    """The keys of `param` the calibrators and validators read, in order."""
    return list(dict.fromkeys(exp.get_key(param) for exp in cal_exposures + val_exposures))


for param in ("fourier_coeffs", "zernike_coeffs"):
    keys = read_keys(param)
    model = model.set(param, {k: v for k, v in model.get(param).items() if k in keys})
model = model.set("defocus", {key: best_state["defocus"][key] for key in read_keys("defocus")})
if dither_flag:  # the constants of the positions the exposures sit at (the (0, 0) position has none)
    for param in ("dither_rotation", "dither_defocus"):
        model = model.set(param, {key: best_state[param][key] for key in model.get(param)})

ff = model.FF
ff -= np.median(ff) - 1.0
ff = ff.at[badpix_bool].set(1.0)
model = model.set("FF", ff)
model = model.set("non_linearity", model.non_linearity.at[:, badpix_bool].set(0))
model = model.set("dark_current", model.dark_current.at[badpix_bool].set(np.median(model.dark_current)))

spectra = {
    **model.spectra,
    "HD-41094_F380M": np.array(-0.12454516),
    "HD-41094_F430M": np.array(-0.14548536),
    "HD-41094_F480M": np.array(-0.12436274),
}
if binary_flag:
    spectra.update({  # [A, B]
        "V-EZ-Aqr_F380M": np.array([-0.07376706, -0.07606107]),
        "V-EZ-Aqr_F430M": np.array([-0.14906244, -0.14530317]),
        "V-EZ-Aqr_F480M": np.array([-0.12971859, -0.12976371]),
    })
model = model.set("spectra", spectra)

if binary_flag:
    faint_A = {k: round(float(v), 4) for k, v in model.contrasts.items() if float(v) < 0.5}
    if faint_A:
        raise ValueError(f"binary A is the fainter star (contrast < 0.5) in {faint_A}: run convert_state_binary.py on the state")

# Carried into the saved states although this run does not fit them: the holes' common shift (held fixed in the
# aperture) and, without the binary, its parameters and focus keys
carry_params = {"hole_shift": best_state["hole_shift"]}
if not binary_flag:
    carry_params.update({key: best_state[key] for key in ("pas", "separations", "contrasts") if key in best_state})
    carry_params["defocus"] = {k: v for k, v in best_state["defocus"].items() if k not in model.defocus}

## Checking the starting fit
With `init_plots`, the data, model and residuals of every exposure at the start (saved in `init_*` folders when run
as a script).

In [ ]:
from amigo.plotting import summarise_fit

if init_plots:
    for name, exps in [("cal", cal_exposures), ("val", val_exposures), ("flat", flat_exposures), ("dark", dark_exposures)]:
        if not exps:
            continue
        this_save_path = None
        if save_flag:
            this_save_path = os.path.join(save_path, f"init_{name}")
            os.mkdir(this_save_path)
        for exp in exps:
            exp.print_summary()
            summarise_fit(model, exp, save_path=this_save_path)

# Loss, Fishers and optimisers
The loss of an exposure is the mean over its pixels of each pixel's chi-squared (residual over noise, squared and
summed over its slopes; `retrain_fns.loss_fn`), plus two priors: the flat field stays near 1 (3.5% rms) and the
non-linearity near its mean (2.5% rms). Training minimises the sum over exposures by gradient descent.

Two functions adjust each step:

- `grads_fn`, applied to the gradients: the calibrators' spectral slopes stay fixed, and bad pixels keep their
  dark current;
- `norm_fn`, applied to the parameters after every update: the flat field is renormalised to median 1, bad pixels
  are reset, and spectral slopes are kept within +-0.8.

In [ ]:
from retrain_fns import ff_reg, nl_reg

reg_dict = {"FF": (1.0, ff_reg), "NL": (1.0, nl_reg)}
cal_stars = ["HD-41094", "V-EZ-Aqr"]


def grads_fn(model, grads, args):
    grad_params = grads.params
    if "spectra" in grad_params.keys():
        for key, val in grad_params["spectra"].items():
            if any(star in key for star in cal_stars):
                grad_params["spectra"][key] = jtu.map(lambda x: 0 * x, val)
    if "dark_current" in grad_params.keys():
        grad_params["dark_current"] = grad_params["dark_current"].at[badpix_bool].set(0)
    return grads.set("params", grad_params), args


def norm_fn(model_params, args):
    if hasattr(model_params, "FF"):
        FF = model_params.FF
        FF -= np.median(FF) - 1.0
        FF = FF.at[badpix_bool].set(1.0)
        model_params = model_params.set("FF", FF)
    if hasattr(model_params, "non_linearity"):
        model_params = model_params.set("non_linearity", model_params.non_linearity.at[:, badpix_bool].set(0))
    if hasattr(model_params, "spectra"):
        model_params = model_params.set("spectra", jtu.map(lambda x: np.clip(x, -0.8, 0.8), model_params.spectra))
    return model_params, args

How strongly the data constrain a parameter varies by many orders of magnitude between parameters, so one step
size cannot suit them all. The trainer therefore divides every gradient by the diagonal of that parameter's
Fisher information matrix (the curvature of the loss, summed over the exposures sharing the key): a learning rate
of 1 is then about one Newton step, and the learning rates below are fractions of it. The network and the pixel
maps have no Fisher; their learning rates are plain step sizes.

Each parameter has its own optimiser. `sgd(lr, start)` is gradient descent with Nesterov momentum 0.6, switched on
at epoch `start`: the per-exposure parameters start first, the shared optics a few epochs later and the pixel maps
last. The network uses Adam, as in the AMIGO paper, with a learning rate that rises from 0 to 1e-4 along half a
cosine over the first 100 epochs. With validators the network is updated after every calibrator and flat batch
(`n_batch` times per epoch), otherwise once per epoch.

To freeze a parameter, comment out its line: it then gets no updates and no Fisher. The detector's single blur
`sigma` is frozen this way, because every exposure uses its filter's blur (`sigmas`) instead.

In [ ]:
from amigo.legacy.fitting import sgd
from retrain_fns import cosine_warmup

nn_lr, nn_warmup_epochs = 1e-4, 100
nn_updates_per_epoch = n_batch if val_flag else 1


def nn_learning_rate(count):
    return nn_lr * cosine_warmup(count / nn_updates_per_epoch, 0.0, nn_warmup_epochs)


optimisers = {
    "fluxes": sgd(3.5e-2, 0),
    "flat_coeffs": sgd(2.5e-2, 0),
    "positions": sgd(3.5e-2, 2),
    "ptt_coeffs": sgd(2.35e-3, 5),
    "fourier_coeffs": sgd(2.35e-3, 5),
    "zernike_coeffs": sgd(2.35e-3, 5),
    "pas": sgd(3.5e-3, 5),
    "separations": sgd(1.75e-3, 7),
    "contrasts": sgd(7e-3, 8),
    "nn_weights": optax.chain(optax.scale_by_adam(), optax.scale_by_learning_rate(nn_learning_rate)),
    "defocus": sgd(1.175e-2, 25),
    "hole_positions": sgd(2.35e-3, 30),
    "dark_current": sgd(3e-2, 35),
    "hole_distortion": sgd(2e-3, 40),
    # "sigma": sgd(7.2e-3, 50),
    "non_linearity": sgd(1e-4, 100),
    "FF": sgd(2e-5, 150),
    "spectra": sgd(4.5e-2, 20),
    "rotations": sgd(2e-2, 2),
    "sigmas": sgd(2e-2, 2),
    "dither_rotation": sgd(2e-2, 2),
    "dither_defocus": sgd(1.175e-2, 25),
}
if not dither_flag:
    for p in ("dither_rotation", "dither_defocus"):
        optimisers.pop(p)
# parameters that only the left-out data would constrain
if not binary_flag:
    for p in ("pas", "separations", "contrasts"):
        optimisers.pop(p, None)
if not flat_flag:
    for p in ("flat_coeffs", "non_linearity", "FF"):
        optimisers.pop(p, None)
if not dark_flag:
    optimisers.pop("dark_current", None)
if not val_flag:
    optimisers.pop("spectra", None)

print(optimisers.keys())


def fitted(params):
    """The parameters of `params` that have an optimiser."""
    return [p for p in params if p in optimisers]


ts = onp.arange(epochs)
fig, ax = plt.subplots(figsize=(4, 2.5))
ax.plot(ts, nn_lr * cosine_warmup(ts, 0, nn_warmup_epochs))
ax.set(title="Network learning rate", xlabel="Epoch", xlim=(0, epochs))
fig.tight_layout()
if save_flag:
    fig.savefig(os.path.join(save_path, "schedules.png"), dpi=200)
    plt.close(fig)
else:
    plt.show()

Now the trainer and its Fishers. They are computed on a cheaper, linear version of the model (no charge migration
and no PSF upsampling) and cached in `fisher_cache`, so a run only computes the ones it does not find there.
Cached Fishers named before amigo's renaming (`fourier_opd`, `filter_opd`, `aberrations`) are hard-linked to the
new names (the same parameters; a link adds no file), so they are not recomputed.

In [ ]:
from amigo.legacy.calibration import ValBatchedTrainer, Trainer
from retrain_fns import loss_fn, aux_fn, looper_fn, batch_looper_fn, summarise_fn

for folder in glob(os.path.join(fisher_cache, "fishers", "*")):
    for old, new in OLD_WF_NAMES.items():
        old_file, new_file = os.path.join(folder, f"{old}.npy"), os.path.join(folder, f"{new}.npy")
        if os.path.exists(old_file) and not os.path.exists(new_file):
            os.link(old_file, new_file)

history_stride = 5  # parameter histories are recorded every 5th epoch (the network's after every batch)
trainer_kwargs = dict(
    loss_fn=loss_fn,
    grad_fn=grads_fn,
    norm_fn=norm_fn,
    summarise_fn=summarise_fn,
    intermediate_prints=[epochs // 2],  # one interim summary, half way through
    save_path=save_path,
    cache=fisher_cache,
    history_stride=history_stride,
    batch_history_max_len=n_batch * 200,  # the network's per-batch history: its last 200 epochs
)
if val_flag:
    trainer = ValBatchedTrainer(**trainer_kwargs, aux_fn=aux_fn, looper_fn=looper_fn)
else:
    trainer = Trainer(**trainer_kwargs, looper_fn=batch_looper_fn)

linear_model = model.set(["psf_upsample", "bleed"], [1, False])
fisher_kwargs = dict(recalculate=recalculate, overwrite=recalculate)
if flat_flag:
    print("Flats")
    trainer = trainer.update_fishers(
        linear_model, flat_exposures, parameters=fitted(["flat_coeffs", "fluxes"]), **fisher_kwargs
    )
if val_flag:
    print("Validators")
    trainer = trainer.update_fishers(
        linear_model,
        val_exposures,
        batch_size=10,
        parameters=fitted(["positions", "ptt_coeffs", "fluxes", "spectra", "defocus", "rotations"]),
        **fisher_kwargs,
    )
print("Calibrators, shared optics")
trainer = trainer.update_fishers(
    linear_model,
    cal_exposures,
    batch_size=10,
    parameters=fitted(
        ["ptt_coeffs", "defocus", "fourier_coeffs", "zernike_coeffs", "hole_positions", "hole_distortion",
         "dither_rotation", "dither_defocus"]
    ),
    **fisher_kwargs,
)
print("Calibrators, per exposure")
trainer = trainer.update_fishers(
    linear_model,
    cal_exposures,
    parameters=fitted(["positions", "fluxes", "spectra", "sigma", "rotations", "sigmas"]),
    **fisher_kwargs,
)
if binary_flag:
    print("Binary")
    trainer = trainer.update_fishers(
        linear_model, calbin_exposures, parameters=fitted(["pas", "separations", "contrasts"]), **fisher_kwargs
    )

# Training
Every epoch goes once through all batches, in a random order drawn from the key in `args` (so the same order in
every run). For each batch the trainer computes the loss and its gradients, divides them by the Fishers and applies
`grads_fn`. The network is updated after each calibrator and flat batch; every other parameter once per epoch, from
the gradients summed over the epoch, followed by `norm_fn`. The epoch with the lowest mean validator loss is kept
as the best state.

In [ ]:
summarise_kwargs = {
    "val_flag": val_flag,
    "flat_flag": flat_flag,
    "binary_flag": binary_flag,
    "dark_flag": dark_flag,
    "save_flag": save_flag,
    "cal_exposures": cal_exposures,
    "val_exposures": val_exposures,
    "flat_exposures": flat_exposures,
    "dark_exposures": dark_exposures,
    "calpsf_exposures": calpsf_exposures,
    "calbin_exposures": calbin_exposures,
    "badpix": badpix,
    "n_batch": n_batch,
    "optimisers": optimisers,
    "amigo_files_path": None if save_flag else scratch_path,  # where interactive runs' scratch states go
    "history_stride": history_stride,
    "carry_params": carry_params,
}


# every non-integer array in the model as float64 (this avoids recompiles)
def to_float(x):
    if isinstance(x, jax.Array) and "i" not in x.dtype.str:
        return np.array(x, dtype=float)
    return x


model = jtu.map(to_float, model)

args = {"key": jr.key(0), "reg_dict": reg_dict}
train_kwargs = dict(
    model=model,
    optimisers=optimisers,
    epochs=epochs,
    batches={**cal_batch, **val_batch, **flat_batch},
    args=args,
    summarise_kwargs=summarise_kwargs,
)
if val_flag:
    train_kwargs.update(
        batched_params=["nn_weights"] if "nn_weights" in optimisers else [],  # updated after every batch
        validators=val_batch,
        validator_params=fitted(["positions", "fluxes", "ptt_coeffs", "spectra", "defocus", "rotations"]),
    )

result = trainer.train(**train_kwargs)

# Saving
`summarise_fn` saves `best_state.npy` (the epoch with the lowest validator loss) and `final_state.npy` (the network
averaged over the last epoch's batches), the parameter histories, and figures of the losses, every exposure's fit,
the wavefronts and the detector maps. The same summary is made half way through (an `epoch_*` folder).

Run as a script, all of it goes to the run folder. Run interactively, the figures are shown and the states are
written to `output_path/scratch` as `scratch_best_state.npy`, `scratch_final_state.npy` and
`scratch_param_history.npy` (by the interim summary too), replacing the previous interactive run's.

In [ ]:
summarise_fn(result, save_path, **summarise_kwargs)